In [31]:
import os
import json
import pandas as pd
import traceback


In [32]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [33]:

load_dotenv()




True

In [34]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    api_key=os.getenv("OPENAI_API_KEY")
)

Project Setup – Imports Explanation

-ChatOpenAI
This is the language model used to generate responses (e.g., MCQs).

-ChatPromptTemplate
Helps create structured prompts with placeholders (like {text} or {topic}).

-PyPDF2
Used to read and extract text from PDF files.

-langchain_community (optional)
Provides additional tools like callbacks (e.g., get_openai_callback) for tracking token usage and cost.

In [35]:
!pip install langchain-community

In [36]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.callbacks import get_openai_callback
import PyPDF2 



quiz_template = """
Text:
{text}

You are an expert MCQ maker.
Given the above text, create {number} multiple choice questions
for {subject} students in a {tone} tone.

Make sure the questions:
- are not repeated
- match the text
- follow this JSON format

RESPONSE_JSON:
{response_json}
"""

quiz_prompt = ChatPromptTemplate.from_template(quiz_template)
quiz_chain = quiz_prompt | llm

review_template = """
You are an expert English grammarian and writer.

Given the following multiple choice quiz for {subject} students:

Quiz_MCQs:
{quiz}

Evaluate the complexity of the quiz in at most 50 words.
If needed, improve the wording and adjust the tone to fit the students' level.
"""

review_prompt = ChatPromptTemplate.from_template(review_template)
review_chain = review_prompt | llm

In [37]:
response_json = """
{
  "1": {
    "mcq": "question here",
    "options": {
      "a": "option a",
      "b": "option b",
      "c": "option c",
      "d": "option d"
    },
    "correct": "a"
  }
}
"""

quiz_result = quiz_chain.invoke({
    "text": "Your extracted PDF text goes here.",
    "number": 5,
    "subject": "biology",
    "tone": "simple",
    "response_json": response_json
})

review_result = review_chain.invoke({
    "subject": "biology",
    "quiz": quiz_result.content
})

print("Generated Quiz:\n")
print(quiz_result.content)

print("\nReview:\n")
print(review_result.content)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}